# Data Integration

## Import and prepare data traffic accident data

### Library and Settings

In [26]:
# Libraries
import os
import re
import fnmatch
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from io import BytesIO
import zipfile
import sqlite3
from pathlib import Path


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

c:\Users\walte\Documents\5semesterprojekte\Data Analytics\swiss-road-accident-severity-\nootebooks


### Importing and calling traffic accidents data

In [27]:
# Check data type and format
url = (
    "https://data.geo.admin.ch/api/stac/v1/"
    "collections/ch.astra.unfaelle-personenschaeden_alle/items"
)

response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()

for item in data["features"]:
    print("Datensatz:", item["id"])

    for name, asset in item.get("assets", {}).items():
        print("Datei:", name)
        print("Format:", asset.get("type"))
        print("URL:", asset.get("href"))

url = "https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle_2056.csv.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()
# 3. Open ZIP and find CSV
with zipfile.ZipFile(BytesIO(response.content)) as z:
    csv_file = next(
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    )

    # 4. Inspect first 10 lines
    with z.open(csv_file) as f:
        for i in range(10):
            print(f.readline().decode("utf-8-sig", errors="replace").strip())

    # 5. Read CSV with automatic separator detection
    with z.open(csv_file) as f:
        df = pd.read_csv(
            f,
            sep=None,
            engine="python",
            encoding="utf-8-sig"
        )

# 6. Display results
print("Rows and columns:", df.shape)
print("Column names:", df.columns.tolist())

display(df.head())

Datensatz: unfaelle-personenschaeden_alle
Datei: unfaelle-personenschaeden_alle_2056.csv.zip
Format: application/x.csv+zip
URL: https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle_2056.csv.zip
Datei: unfaelle-personenschaeden_alle_2056.xtf.zip
Format: application/x.interlis+zip; version=2.3
URL: https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle_2056.xtf.zip
Datei: unfaelle-personenschaeden_alle_4326.json.zip
Format: application/x.geojson+zip
URL: https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle_4326.json.zip
Datei: unfaelle-personenschaeden_alle.zip
Format: application/zip
URL: https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle.zip
"AccidentUID","AccidentType","AccidentType_de","Ac

,AccidentUID,AccidentType,AccidentType_de,AccidentType_fr,AccidentType_it,AccidentType_en,AccidentSeverityCategory,AccidentSeverityCategory_de,AccidentSeverityCategory_fr,AccidentSeverityCategory_it,...,AccidentMonth_fr,AccidentMonth_it,AccidentMonth_en,AccidentWeekDay,AccidentWeekDay_de,AccidentWeekDay_fr,AccidentWeekDay_it,AccidentWeekDay_en,AccidentHour,AccidentHour_text
0,9965818AA56DF000E0430A865E33F000,at2,Auffahrunfall,accident par tamponnement,Incidente di tamponamento,Accident with rear-end collision,as2,Unfall mit Schwerverletzten,accident avec blessés graves,Incidente con feriti gravi,...,janvier,Gennaio,January,aw402,Dienstag,mardi,Martedì,Tuesday,12,12h-13h
1,996651D38EC1A0C8E0430A865E33A0C8,at5,Überqueren der Fahrbahn,accident en traversant une route,Incidente nell'attraversare la carreggiata,Accident when crossing the lane(s),as3,Unfall mit Leichtverletzten,accident avec blessés légers,Incidente con feriti leggeri,...,janvier,Gennaio,January,aw403,Mittwoch,mercredi,Mercoledì,Wednesday,7,07h-08h
2,99797FE697E750A4E0430A865E3350A4,at4,Einbiegeunfall,accident en s'engageant sur une route,Incidente nell’immettersi in una strada,Accident when turning-into main road,as2,Unfall mit Schwerverletzten,accident avec blessés graves,Incidente con feriti gravi,...,janvier,Gennaio,January,aw402,Dienstag,mardi,Martedì,Tuesday,11,11h-12h
3,997B044BBD5660F6E0430A865E3360F6,at8,Fussgängerunfall,accident impliquant des piétons,Incidente con pedoni,Accident involving pedestrian(s),as3,Unfall mit Leichtverletzten,accident avec blessés légers,Incidente con feriti leggeri,...,janvier,Gennaio,January,aw401,Montag,lundi,Lunedí,Monday,11,11h-12h
4,99AE12B0D5D93060E0430A865E333060,at4,Einbiegeunfall,accident en s'engageant sur une route,Incidente nell’immettersi in una strada,Accident when turning-into main road,as3,Unfall mit Leichtverletzten,accident avec blessés légers,Incidente con feriti leggeri,...,janvier,Gennaio,January,aw402,Dienstag,mardi,Martedì,Tuesday,18,18h-19h


### Inspecting and choosing relevant data columns 


In [ ]:
# 1. Keep original dataset unchanged
df_raw = df.copy()

# 2. Select relevant columns for our project
columns = [
    "AccidentUID",
    "AccidentYear",
    "AccidentMonth",
    "AccidentWeekDay",
    "AccidentHour",
    "AccidentSeverityCategory",
    "AccidentType",
    "RoadType",
    "MunicipalityCode",
    "CantonCode",
    "AccidentLocation_CHLV95_E",
    "AccidentLocation_CHLV95_N",
    "AccidentInvolvingPedestrian",
    "AccidentInvolvingBicycle",
    "AccidentInvolvingMotorcycle"
]

# 3. Create working dataset
df_accidents = df_raw[columns].copy()

# 4. Filter years 2021–2025
df_accidents["AccidentYear"] = pd.to_numeric(
    df_accidents["AccidentYear"],
    errors="coerce"
)

df_accidents = df_accidents[
    df_accidents["AccidentYear"].between(2021, 2025)
].copy()

# 5. Reset index
df_accidents.reset_index(drop=True, inplace=True)

# 6. Inspect dataset
print("Original dataset:", df_raw.shape)
print("Filtered dataset:", df_accidents.shape)

print("\nAccidents per year:")
print(df_accidents["AccidentYear"].value_counts().sort_index())

print("\nSeverity categories:")
print(df_accidents["AccidentSeverityCategory"].value_counts(dropna=False))

print("\nAccident hours:")
print(df_accidents["AccidentHour"].value_counts(dropna=False).sort_index())

print("\nMissing values:")
print(df_accidents.isnull().sum())

print("\nDuplicate accident IDs:")
print(df_accidents["AccidentUID"].duplicated().sum())

display(df_accidents.head(10))

Original dataset: (267761, 36)
Filtered dataset: (89544, 15)

Accidents per year:
AccidentYear
2021    17436
2022    18396
2023    18254
2024    17432
2025    18026
Name: count, dtype: int64

Severity categories:
AccidentSeverityCategory
as3    69774
as2    18674
as1     1096
Name: count, dtype: int64

Accident hours:
AccidentHour
0     1193
1      987
2      785
3      634
4      690
5     1060
6     3422
7     5195
8     3909
9     3453
10    3979
11    5079
12    5215
13    5713
14    5765
15    6500
16    8024
17    9300
18    6252
19    4064
20    2588
21    2208
22    1961
23    1568
Name: count, dtype: int64

Missing values:
AccidentUID                    0
AccidentYear                   0
AccidentMonth                  0
AccidentWeekDay                0
AccidentHour                   0
AccidentSeverityCategory       0
AccidentType                   0
RoadType                       0
MunicipalityCode               0
CantonCode                     0
AccidentLocation_CHLV95_E     

,AccidentUID,AccidentYear,AccidentMonth,AccidentWeekDay,AccidentHour,AccidentSeverityCategory,AccidentType,RoadType,MunicipalityCode,CantonCode,AccidentLocation_CHLV95_E,AccidentLocation_CHLV95_N,AccidentInvolvingPedestrian,AccidentInvolvingBicycle,AccidentInvolvingMotorcycle
0,00060284C00AC34F87BBABA6C3809CCD,2024,11,aw404,17,as3,at3,rt433,96,ZH,2677917,1255472,False,True,False
1,00075447B4B5C04CAE5F154F8D369CAC,2024,8,aw403,12,as2,at1,rt432,5451,VD,2571182,1190763,False,False,True
2,00098A69568701B0E0630AB38B02C0F4,2023,4,aw407,22,as2,at2,rt430,4107,AG,2660049,1255757,False,False,True
3,00098A695D7B01B0E0630AB38B02C0F4,2023,6,aw405,17,as3,at5,rt432,261,ZH,2683261,1252404,False,False,False
4,00098A695E4501B0E0630AB38B02C0F4,2023,6,aw405,8,as2,at3,rt432,4048,AG,2669529,1255399,False,True,False
5,00098A695FB501B0E0630AB38B02C0F4,2023,6,aw401,6,as3,at2,rt430,4024,AG,2662742,1256350,False,False,False
6,00098A6964D801B0E0630AB38B02C0F4,2023,6,aw402,11,as3,at3,rt433,942,BE,2614682,1178262,False,True,False
7,00098A696C0C01B0E0630AB38B02C0F4,2023,7,aw403,12,as3,at4,rt432,3955,GR,2761655,1203871,False,False,False
8,00098A696FB701B0E0630AB38B02C0F4,2023,6,aw407,21,as3,at0,rt432,1507,NW,2666536,1205218,False,True,False
9,00098A69859C01B0E0630AB38B02C0F4,2023,5,aw401,9,as3,at4,rt433,733,BE,2587213,1218927,False,False,False


### Cleaning ASTRA traffic accident data

In [29]:
# 1. Create a copy of our filtered ASTRA dataset
df_clean = df_accidents.copy()

print("Original records:", len(df_clean))

# 2. Check data types
print("\n--- DATA TYPES ---")
print(df_clean.dtypes)

# 3. Check missing values
print("\n--- MISSING VALUES ---")
missing = pd.DataFrame({
    "Missing": df_clean.isna().sum(),
    "Percentage": (df_clean.isna().mean() * 100).round(2)
})
display(missing)

# 4. Check duplicates
print("\n--- DUPLICATES ---")
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate AccidentUIDs:",
      df_clean["AccidentUID"].duplicated().sum())

# 5. Check severity categories
print("\n--- ACCIDENT SEVERITY ---")
print(df_clean["AccidentSeverityCategory"].value_counts(dropna=False))

# 6. Check accident hours
print("\n--- ACCIDENT HOURS ---")
print(df_clean["AccidentHour"].value_counts(dropna=False).sort_index())

# 7. Check accident and road types
print("\n--- ACCIDENT TYPES ---")
print(df_clean["AccidentType"].value_counts(dropna=False))

print("\n--- ROAD TYPES ---")
print(df_clean["RoadType"].value_counts(dropna=False))

# 8. Standardize municipality codes for the BFS join
df_clean["MunicipalityCode"] = (
    df_clean["MunicipalityCode"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

print("\n--- MUNICIPALITY CODES ---")
print("Missing municipality codes:",
      df_clean["MunicipalityCode"].isna().sum())

print("Unique municipalities:",
      df_clean["MunicipalityCode"].nunique())

# 9. Validate year and hour values
year = pd.to_numeric(df_clean["AccidentYear"], errors="coerce")
hour = pd.to_numeric(df_clean["AccidentHour"], errors="coerce")

print("\n--- VALIDATION ---")
print("Invalid years:", (~year.between(2021, 2025)).sum())
print("Invalid or missing hours:", (~hour.between(0, 23)).sum())

# 10. Final check
print("\n--- SUMMARY ---")
print("Records before cleaning:", len(df_accidents))
print("Records after basic cleaning:", len(df_clean))
print("Columns:", len(df_clean.columns))

display(df_clean.head())


Original records: 89544

--- DATA TYPES ---
AccidentUID                    object
AccidentYear                    int64
AccidentMonth                   int64
AccidentWeekDay                object
AccidentHour                    int64
AccidentSeverityCategory       object
AccidentType                   object
RoadType                       object
MunicipalityCode                int64
CantonCode                     object
AccidentLocation_CHLV95_E       int64
AccidentLocation_CHLV95_N       int64
AccidentInvolvingPedestrian      bool
AccidentInvolvingBicycle         bool
AccidentInvolvingMotorcycle      bool
dtype: object

--- MISSING VALUES ---


,Missing,Percentage
AccidentUID,0,0.0
AccidentYear,0,0.0
AccidentMonth,0,0.0
AccidentWeekDay,0,0.0
AccidentHour,0,0.0
AccidentSeverityCategory,0,0.0
AccidentType,0,0.0
RoadType,0,0.0
MunicipalityCode,0,0.0
CantonCode,0,0.0



--- DUPLICATES ---
Duplicate rows: 0
Duplicate AccidentUIDs: 0

--- ACCIDENT SEVERITY ---
AccidentSeverityCategory
as3    69774
as2    18674
as1     1096
Name: count, dtype: int64

--- ACCIDENT HOURS ---
AccidentHour
0     1193
1      987
2      785
3      634
4      690
5     1060
6     3422
7     5195
8     3909
9     3453
10    3979
11    5079
12    5215
13    5713
14    5765
15    6500
16    8024
17    9300
18    6252
19    4064
20    2588
21    2208
22    1961
23    1568
Name: count, dtype: int64

--- ACCIDENT TYPES ---
AccidentType
at0     32706
at2     15101
at4     10454
at8      9564
at3      6949
at1      4382
at5      4279
at6      3584
at7      1124
at00     1015
at9       386
Name: count, dtype: int64

--- ROAD TYPES ---
RoadType
rt432    43133
rt433    34452
rt430     7467
rt439     3755
rt431      701
rt434       36
Name: count, dtype: int64

--- MUNICIPALITY CODES ---
Missing municipality codes: 0
Unique municipalities: 2039

--- VALIDATION ---
Invalid years: 0
Invalid

,AccidentUID,AccidentYear,AccidentMonth,AccidentWeekDay,AccidentHour,AccidentSeverityCategory,AccidentType,RoadType,MunicipalityCode,CantonCode,AccidentLocation_CHLV95_E,AccidentLocation_CHLV95_N,AccidentInvolvingPedestrian,AccidentInvolvingBicycle,AccidentInvolvingMotorcycle
0,00060284C00AC34F87BBABA6C3809CCD,2024,11,aw404,17,as3,at3,rt433,96,ZH,2677917,1255472,False,True,False
1,00075447B4B5C04CAE5F154F8D369CAC,2024,8,aw403,12,as2,at1,rt432,5451,VD,2571182,1190763,False,False,True
2,00098A69568701B0E0630AB38B02C0F4,2023,4,aw407,22,as2,at2,rt430,4107,AG,2660049,1255757,False,False,True
3,00098A695D7B01B0E0630AB38B02C0F4,2023,6,aw405,17,as3,at5,rt432,261,ZH,2683261,1252404,False,False,False
4,00098A695E4501B0E0630AB38B02C0F4,2023,6,aw405,8,as2,at3,rt432,4048,AG,2669529,1255399,False,True,False


### Merge bfs municipality data with accident traffic data

In [30]:
bfs = pd.read_csv("../data/raw/stadt_land.csv", sep=";", skiprows=[0, 2])

bfs = bfs.iloc[:, [0, 6]]
bfs.columns = ["MunicipalityCode", "urban_rural"]

bfs["MunicipalityCode"] = pd.to_numeric(bfs["MunicipalityCode"])
df_clean["MunicipalityCode"] = pd.to_numeric(df_clean["MunicipalityCode"])

df_merged = df_clean.merge(bfs, on="MunicipalityCode", how="left", validate="many_to_one")

print("Match rate:", round(df_merged["urban_rural"].notna().mean() * 100, 2), "%")
display(df_merged.head())

Match rate: 99.61 %


,AccidentUID,AccidentYear,AccidentMonth,AccidentWeekDay,AccidentHour,AccidentSeverityCategory,AccidentType,RoadType,MunicipalityCode,CantonCode,AccidentLocation_CHLV95_E,AccidentLocation_CHLV95_N,AccidentInvolvingPedestrian,AccidentInvolvingBicycle,AccidentInvolvingMotorcycle,urban_rural
0,00060284C00AC34F87BBABA6C3809CCD,2024,11,aw404,17,as3,at3,rt433,96,ZH,2677917,1255472,False,True,False,1.0
1,00075447B4B5C04CAE5F154F8D369CAC,2024,8,aw403,12,as2,at1,rt432,5451,VD,2571182,1190763,False,False,True,3.0
2,00098A69568701B0E0630AB38B02C0F4,2023,4,aw407,22,as2,at2,rt430,4107,AG,2660049,1255757,False,False,True,2.0
3,00098A695D7B01B0E0630AB38B02C0F4,2023,6,aw405,17,as3,at5,rt432,261,ZH,2683261,1252404,False,False,False,1.0
4,00098A695E4501B0E0630AB38B02C0F4,2023,6,aw405,8,as2,at3,rt432,4048,AG,2669529,1255399,False,True,False,1.0


### Adding binary variable severe and night 

In [33]:
df_final = df_merged.rename(columns={
    "AccidentUID": "accident_id",
    "AccidentYear": "year",
    "AccidentHour": "hour",
    "AccidentSeverityCategory": "severity",
    "AccidentType": "accident_type",
    "RoadType": "road_type",
    "MunicipalityCode": "municipality_code"
}).copy()

df_final["severe"] = df_final["severity"].map({
    "as1": 1, "as2": 1, "as3": 0
})

df_final["hour"] = pd.to_numeric(df_final["hour"], errors="coerce")

df_final["night"] = (
    (df_final["hour"] >= 22) | (df_final["hour"] < 6)
).astype("Int64")

df_final.loc[~df_final["hour"].between(0, 23), "night"] = pd.NA

df_final["area_type"] = df_final["urban_rural"].map({
    1: "Urban",
    2: "Intermediate",
    3: "Rural"
})

df_final = df_final.drop(columns=["_merge"], errors="ignore")

print(df_final["area_type"].value_counts(dropna=False))
print(df_final[["severe", "night", "area_type"]].isna().sum())

display(df_final.head())

area_type
Urban           58336
Intermediate    17463
Rural           13394
NaN               351
Name: count, dtype: int64
severe         0
night          0
area_type    351
dtype: int64


,accident_id,year,AccidentMonth,AccidentWeekDay,hour,severity,accident_type,road_type,municipality_code,CantonCode,AccidentLocation_CHLV95_E,AccidentLocation_CHLV95_N,AccidentInvolvingPedestrian,AccidentInvolvingBicycle,AccidentInvolvingMotorcycle,urban_rural,severe,night,area_type
0,00060284C00AC34F87BBABA6C3809CCD,2024,11,aw404,17,as3,at3,rt433,96,ZH,2677917,1255472,False,True,False,1.0,0,0,Urban
1,00075447B4B5C04CAE5F154F8D369CAC,2024,8,aw403,12,as2,at1,rt432,5451,VD,2571182,1190763,False,False,True,3.0,1,0,Rural
2,00098A69568701B0E0630AB38B02C0F4,2023,4,aw407,22,as2,at2,rt430,4107,AG,2660049,1255757,False,False,True,2.0,1,1,Intermediate
3,00098A695D7B01B0E0630AB38B02C0F4,2023,6,aw405,17,as3,at5,rt432,261,ZH,2683261,1252404,False,False,False,1.0,0,0,Urban
4,00098A695E4501B0E0630AB38B02C0F4,2023,6,aw405,8,as2,at3,rt432,4048,AG,2669529,1255399,False,True,False,1.0,1,0,Urban


### Store final data in sql database

In [34]:
print("Records:", len(df_final))
print("Duplicates:", df_final["accident_id"].duplicated().sum())

print("\nMissing values:")
print(df_final[["severe", "night", "area_type"]].isna().sum())

df_final = df_final.drop(columns=["_merge"], errors="ignore")

Path("../data/processed").mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect("../data/processed/accidents.db")
df_final.to_sql("accidents", conn, if_exists="replace", index=False)

result = pd.read_sql("""
    SELECT year, COUNT(*) AS accidents
    FROM accidents
    GROUP BY year
    ORDER BY year
""", conn)

conn.close()

display(result)

Records: 89544
Duplicates: 0

Missing values:
severe         0
night          0
area_type    351
dtype: int64


,year,accidents
0,2021,17436
1,2022,18396
2,2023,18254
3,2024,17432
4,2025,18026


### Sql-queries and JOIN-query

In [ ]:
conn = sqlite3.connect("../data/processed/accidents.db")

q1 = pd.read_sql("""
    SELECT year, COUNT(*) AS accidents
    FROM accidents
    WHERE year >= 2021
    GROUP BY year
""", conn)

q2 = pd.read_sql("""
    SELECT area_type, COUNT(*) AS severe_accidents
    FROM accidents
    WHERE severe = 1 AND area_type IS NOT NULL
    GROUP BY area_type
""", conn)

bfs.to_sql("municipalities", conn, if_exists="replace", index=False)

# JOIN-Query
q3 = pd.read_sql("""
    SELECT b.urban_rural, COUNT(*) AS accidents
    FROM accidents a
    JOIN municipalities b
    ON a.municipality_code = b.MunicipalityCode
    GROUP BY b.urban_rural
""", conn)

display(q1)
display(q2)
display(q3)

conn.close()

,year,accidents
0,2021,17436
1,2022,18396
2,2023,18254
3,2024,17432
4,2025,18026


,area_type,severe_accidents
0,Intermediate,4278
1,Rural,3805
2,Urban,11607


,urban_rural,accidents
0,1,58336
1,2,17463
2,3,13394


### Data validation and parquet

In [40]:
df_model = df_final.dropna(
    subset=["severe", "night", "area_type", "accident_type", "road_type"]
).copy()

print("Original:", len(df_accidents))
print("BFS matched:", df_final["area_type"].notna().sum())
print("Final:", len(df_model))
print("Excluded:", len(df_accidents) - len(df_model))
print("Data loss (%):", round((1 - len(df_model) / len(df_accidents)) * 100, 2))

print(df_model[["severe", "night", "area_type"]].isna().sum())

df_model.to_parquet("../data/processed/accidents_clean.parquet", index=False)

Original: 89544
BFS matched: 89193
Final: 89193
Excluded: 351
Data loss (%): 0.39
severe       0
night        0
area_type    0
dtype: int64


In [41]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 11
Datetime: 2026-11-10 16:24:57
Python Version: 3.13.5
-----------------------------------
